# **PREDICCIÓN DE LA PREMIER LEAGUE: SECUENCIAL VS. SPARK**

## **1. MODELO DIXON-COLES**

---

### **1.1. MOTIVACIÓN DEL PROBLEMA**

Predecir resultados de partidos de fútbol es un problema estadístico clásico. El objetivo es estimar la probabilidad de que ocurra cada posible marcador (e.g., 2-1, 0-0, 3-2, ...) dado un par de equipos. Con estas probabilidades podemos derivar las probabilidades de victoria local, empate y victoria visitante.

Los datos que necesitamos son los resultados históricos de partidos: quién jugó en casa, quién visitó, y cuántos goles marcó cada equipo. A partir de estos datos, queremos construir un modelo que capture:

- La **fortaleza atacante** de cada equipo (cuántos goles tiende a marcar)
- La **fortaleza defensiva** de cada equipo (cuántos goles tiende a encajar)
- La **ventaja de jugar en casa** (los equipos locales suelen marcar más)

La dificultad está en que no todos los equipos juegan entre sí con igual frecuencia, y la calidad de los equipos varía mucho. Además, los equipos cambian a lo largo del tiempo: fichajes, lesiones, cambios de entrenador, etc.

---

### **1.2. EL MODELO BÁSICO DE POISSON**

El número de goles marcados por un equipo en un partido sigue aproximadamente una **distribución de Poisson**. Esta distribución modela el número de eventos que ocurren en un intervalo de tiempo dado que ocurren a una tasa media constante. En una liga, todos los partidos son de 90 minutos + pequeños tiempos de descuento en general similares, esta hipótesis es fundamental, ya que no serían variables comparables si hubiera prórrogas donde se llega a 120 minutos.

La probabilidad de que un equipo marque exactamente *x* goles es:

$$P(X = x) = \frac{e^{-\lambda} \lambda^x}{x!}, \quad \lambda > 0$$

donde $\lambda$ es la tasa media de goles (el parámetro que el modelo estima).



En el **modelo Poisson básico** (BP), los goles del equipo local y del visitante se tratan como **dos variables independientes**:

$$P(X_{i,j} = x, Y_{j,i} = y) = \frac{e^{-\lambda} \lambda^x}{x!} \cdot \frac{e^{-\mu} \mu^y}{y!}$$

donde:
- $\lambda = \alpha_i \cdot \beta_j \cdot \gamma$ → goles esperados del equipo local $i$
- $\mu = \alpha_j \cdot \beta_i$ → goles esperados del equipo visitante $j$
- $\alpha_i$ → fuerza atacante del equipo $i$
- $\beta_i$ → debilidad defensiva del equipo $i$ (mayor = peor defensa)
- $\gamma$ → factor de ventaja local (se asume el mismo para todos los equipos)

En la práctica se trabaja en escala logarítmica:

$$\log(\lambda) = \alpha_i + \beta_j + \gamma, \quad \log(\mu) = \alpha_j + \beta_i$$

### **1.2.1. LIMITACIONES**

El modelo BP tiene un problema conocido: **subestima la frecuencia de resultados con pocos goles** (0-0, 1-0, 0-1, 1-1). Los empates a 0 y los resultados por la mínima aparecen con menos frecuencia de la esperada según el modelo.

---

### **1.3. EL MODELO DE DIXON-COLES**

En su artículo de 1997, **Mark Dixon y Stuart Coles** propusieron dos mejoras sobre el modelo BP:

### **1.3.1 CORRECCIÓN PARA RESULTADOS DE BAJO MARCADOR**

Se introduce una función de corrección $\tau_{\lambda,\mu}(x, y)$ que ajusta las probabilidades de los 4 resultados problemáticos:

$$P(X_{i,j} = x, Y_{j,i} = y) = \tau_{\lambda,\mu}(x, y) \cdot \frac{e^{-\lambda} \lambda^x}{x!} \cdot \frac{e^{-\mu} \mu^y}{y!}$$

donde:

$$\tau_{\lambda,\mu}(x, y) = \begin{cases}
1 - \lambda \mu \rho & \text{si } x=0, y=0 \\
1 + \lambda \rho & \text{si } x=0, y=1 \\
1 + \mu \rho & \text{si } x=1, y=0 \\
1 - \rho & \text{si } x=1, y=1 \\
1 & \text{en otro caso}
\end{cases}$$

El parámetro $\rho$ (rho) controla la intensidad de la corrección. Cuando $\rho = 0$, el modelo es equivalente al BP estándar. En la práctica, $\rho$ suele ser ligeramente negativo (≈ -0.13).

### **1.3.2 PONDERACIÓN TEMPORAL (TIME DECAY)**

Los partidos más recientes deben pesar más que los antiguos. Esto es especialmente relevante cuando se usan varios años de datos: un partido de hace 4 años es menos informativo que uno de hace 2 semanas. Esto ayuda a tener en cuenta fichajes, lesiones, rachas...

La función de ponderación temporal es una **exponencial negativa**:

$$\phi(t) = e^{-\xi \cdot t}$$

donde $t$ es el número de días desde que se jugó el partido (medido hacia atrás desde el momento de predicción), y $\xi$ (xi) controla la velocidad del decaimiento. Con $\xi = 0$ todos los partidos pesan igual. Dixon-Coles sugiere $\xi = 0.0065$, aunque utilizan medias semanas por lo que en el artículo recomienda $\xi = 0.00325$.

---

### **1.4. ESTIMACIÓN DE PARÁMETROS: MÁXIMA VEROSIMILITUD**

Para encontrar los parámetros del modelo ($\alpha_i$, $\beta_i$, $\rho$, $\gamma$) se utiliza **Estimación por Máxima Verosimilitud (MLE)**.

### **1.4.1. LA FUNCIÓN DE LOG-VEROSIMILITUD**

Con $N$ partidos indexados $k = 1, \ldots, N$, la función de log-verosimilitud ponderada es:

$$\ell(\alpha_i, \beta_i, \rho, \gamma) = \sum_{k=1}^{N} e^{-\xi \cdot t_k} \left[ \log \tau_{\lambda_k, \mu_k}(x_k, y_k) + \log P_{\text{Poisson}}(x_k; \lambda_k) + \log P_{\text{Poisson}}(y_k; \mu_k) \right]$$

donde $t_k$ es el número de días que han pasado desde el partido $k$ hasta el momento actual.

### **1.4.2. RESTRICCIÓN DE IDENTIFICABILIDAD**

Para evitar sobreparametrización (el modelo tendría infinitas soluciones equivalentes), se impone la restricción:

$$\frac{1}{n} \sum_i \alpha_i = 1 \quad \Leftrightarrow \quad \sum_i \log(\alpha_i) = 0$$

O en escala logarítmica, que los valores de ataque sumen cero. Esto ancla la escala del modelo y garantiza una solución única.

### **1.4.3. ALGORITMO DE OPTIMIZACIÓN**

Se minimiza el **negativo de la log-verosimilitud** usando el algoritmo `L-BFGS-B` de `scipy.optimize.minimize`, que es eficiente para problemas con muchos parámetros y soporta restricciones de caja (bounds).

---
### **1.5. CARGAR PARÁMETROS**

Los parámetros se estiman en otro cuaderno, se guardan y se cargan aquí. Los parámetros del Ipswich (no aparece en train) se estiman promediando los parámetros de otros clubes en condiciones similares (ascendidos de segunda una sola temporada): Luton, Watford y Norwich.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from scipy.optimize import minimize
from IPython.display import display, HTML
import json
import statistics
import random
import time
from pyspark.sql import SparkSession


In [ ]:
def cargar_params(filepath):
    with open(filepath) as f:
        data = json.load(f)
    params = {}
    for team, vals in data['teams'].items():
        params[f'attack_{team}']  = vals['log_attack']
        params[f'defence_{team}'] = vals['log_defence']
    params['rho']      = data['meta']['rho']
    params['home_adv'] = data['meta']['home_adv']
    return params

Cargamos las estimaciones de parámetros con los datos de entrenamiento de las temporadas 20/21 a 23/24, y con todos los equipos incluyendo la temporada 24/25.

In [ ]:
params2324 = cargar_params('estimaciones_2324.json')
print(f"Rho: {params2324['rho']:.4f}")
print(f"Ventaja local: {params2324['home_adv']:.4f}")

Rho: -0.0100
Ventaja local: 0.2036


In [ ]:
print(params2324)

{'attack_Man City': 0.0854, 'defence_Man City': 0.1132, 'attack_Arsenal': 0.0, 'defence_Arsenal': 0.1444, 'attack_Liverpool': -0.0563, 'defence_Liverpool': 0.4108, 'attack_Newcastle': -0.1002, 'defence_Newcastle': 0.6936, 'attack_Tottenham': -0.201, 'defence_Tottenham': 0.7582, 'attack_Chelsea': -0.2059, 'defence_Chelsea': 0.7043, 'attack_Aston Villa': -0.2568, 'defence_Aston Villa': 0.6953, 'attack_Man United': -0.3905, 'defence_Man United': 0.6321, 'attack_Crystal Palace': -0.4271, 'defence_Crystal Palace': 0.6196, 'attack_West Ham': -0.4274, 'defence_West Ham': 0.8824, 'attack_Brentford': -0.4468, 'defence_Brentford': 0.713, 'attack_Fulham': -0.4597, 'defence_Fulham': 0.695, 'attack_Leicester': -0.469, 'defence_Leicester': 0.783, 'attack_Luton': -0.4763, 'defence_Luton': 1.0989, 'attack_Brighton': -0.4844, 'defence_Brighton': 0.6871, 'attack_Bournemouth': -0.5409, 'defence_Bournemouth': 0.809, 'attack_Leeds': -0.5849, 'defence_Leeds': 0.9876, "attack_Nott'm Forest": -0.5883, "defenc

Como llamar a los parámetros: (recordemos $log(\lambda) = \alpha + \beta + \gamma$)

In [ ]:
team = "Arsenal"

attack = params2324[f"attack_{team}"] # alfa
print(attack)

defence = params2324[f"defence_{team}"] # beta
print(defence)

0.0
0.1444


Cargamos las estimaciones de parámetros con los datos de entrenamiento de las temporadas 20/21 a 24/25, y con todos los equipos incluyendo la temporada 25/26.

In [ ]:
params2425 = cargar_params('estimaciones_2425.json')
print(f"Rho: {params2425['rho']:.4f}")
print(f"Ventaja local: {params2425['home_adv']:.4f}")

Rho: -0.0026
Ventaja local: 0.1118


In [ ]:
print(params2425)

{'attack_Liverpool': 0.1389, 'defence_Liverpool': 0.3619, 'attack_Man City': 0.0652, 'defence_Man City': 0.2268, 'attack_Arsenal': 0.0, 'defence_Arsenal': 0.0968, 'attack_Newcastle': -0.0076, 'defence_Newcastle': 0.4804, 'attack_Brighton': -0.132, 'defence_Brighton': 0.6482, 'attack_Tottenham': -0.1414, 'defence_Tottenham': 0.7652, 'attack_Chelsea': -0.1485, 'defence_Chelsea': 0.3986, 'attack_Brentford': -0.1498, 'defence_Brentford': 0.5895, 'attack_Aston Villa': -0.1816, 'defence_Aston Villa': 0.4881, "attack_Nott'm Forest": -0.2823, "defence_Nott'm Forest": 0.5265, 'attack_Crystal Palace': -0.2832, 'defence_Crystal Palace': 0.5059, 'attack_Bournemouth': -0.288, 'defence_Bournemouth': 0.4963, 'attack_Luton': -0.2929, 'defence_Luton': 1.0093, 'attack_Fulham': -0.2966, 'defence_Fulham': 0.5793, 'attack_Wolves': -0.3629, 'defence_Wolves': 0.7143, 'attack_West Ham': -0.3898, 'defence_West Ham': 0.6992, 'attack_Man United': -0.4049, 'defence_Man United': 0.5512, 'attack_Leeds': -0.4349, 'd

## **2. SIMULACIÓN DE LA PREMIER LEAGUE SECUENCIAL**

### **2.1. DEFINCIÓN DE FUNCIONES**

- Función **simular partido**: Le damos un equipo local y uno visitante y devuelve un resultado aleatorio en función de los parámetros.

Si $log(\lambda)= α+\beta+γ$ entonces $λ= e^{α+\beta+γ}$

In [ ]:
def simular_partido(equipo_local: str, equipo_visitante: str, params: dict) -> str:
    lamb = np.exp(
        params[f'attack_{equipo_local}'] +
        params[f'defence_{equipo_visitante}'] +
        params['home_adv']
    )
    mu = np.exp(
        params[f'attack_{equipo_visitante}'] +
        params[f'defence_{equipo_local}']
    )

    def tau(x, y):
        if   x == 0 and y == 0: return 1 - lamb * mu * params['rho']
        elif x == 0 and y == 1: return 1 + lamb * params['rho']
        elif x == 1 and y == 0: return 1 + mu   * params['rho']
        elif x == 1 and y == 1: return 1 - params['rho']
        else:                   return 1.0

    max_goals = 10
    scores, probs = [], []
    for x in range(max_goals + 1):
        for y in range(max_goals + 1):
            p = tau(x, y) * poisson.pmf(x, lamb) * poisson.pmf(y, mu)
            scores.append((x, y))
            probs.append(p)

    probs = np.array(probs)
    probs /= probs.sum()

    x, y = scores[np.random.choice(len(scores), p=probs)]
    return f'{x}-{y}'



---------------------------------------------------------------------


- Función **estimar partido**: Le damos los dos equipos, local y visitante, y devuelve el resultados más probable (moda) y las probabildiades de victoria, empate o derrota. Esta es la función útil para las apuestas. Esta función se reutiliza en el apartado 4 como referencia (baseline) secuencial frente a su versión paralelizada con Spark.

In [ ]:
def estimar_partido(equipo_local, equipo_visitante, params, MC = 1000):

  resultados = [simular_partido(equipo_local, equipo_visitante, params) for _ in range(MC)]
  victoria_local = 0
  victoria_visitante = 0
  empate = 0
  for marcador in resultados:
    goles_local, goles_visitante = marcador.split("-")
    if int(goles_local) > int(goles_visitante):
      victoria_local += 1
    elif int(goles_local) < int(goles_visitante):
      victoria_visitante += 1
    else:
      empate += 1
  probabilidad_local = round(victoria_local/MC,4)
  probabilidad_visitante = round(victoria_visitante/MC,4)
  probabilidad_empate = round(1 - probabilidad_local - probabilidad_visitante,4)


  modas = statistics.multimode(resultados)
  return str(random.choice(modas)), probabilidad_local, probabilidad_empate, probabilidad_visitante

-----------------------------------------------------------

- Función **simular liga**: Le damos la lista de equipos y simulamos la liga, devuelve un diccionario ordenado según la posición de cada equipo con sus puntos correspondientes.

In [ ]:
def simular_liga(equipos: list, params) -> list:
  """
  Simula todos los enfrentamientos entre equipos y devuelve un diccionario
  con los puntos ordenado de mayor a menor
  """
  partidos = []
  puntos = {}
  for equipo1 in equipos:
    puntos[equipo1] = 0

    for equipo_rival in equipos:
      if equipo1 == equipo_rival:
        pass
      else:
        partidos.append(f'{equipo1} vs {equipo_rival}: {simular_partido(equipo1, equipo_rival, params)}')

  # Ahora hay partidos= ["equipo1 vs equipo_rival: 1-2", ...]
  # Y un diccionario con puntos = {equipo1: 0, equipo2: 0, ...}

  for resultado in partidos:
    partes = resultado.split(' vs ')
    partes = [partes[0]] + partes[1].split(': ')
    marcador = partes[2].split("-")
    if int(marcador[0]) > int(marcador[1]):
      puntos[partes[0]] += 3
    elif int(marcador[0]) < int(marcador[1]):
      puntos[partes[1]] += 3
    else:
      puntos[partes[1]] += 1
      puntos[partes[0]] += 1

  puestos = dict(sorted(puntos.items(), key=lambda item: item[1], reverse=True))

  puestos_final = []  #Lista ordenada con los equipos, el 0 es el priemro el 1 el segundo... etc

  for equipo in puestos.keys():
    puestos_final.append(equipo)

  return puestos, puestos_final

- Función **estimar liga**: Dada una lista de 20 equipos. Estima para cada equipo, las probabilidades de terminar la liga en la posición i-ésima para $i = 1, ..., 20$

In [ ]:
def estimar_liga(B:int=1000, lista_equipos: list=[], params={}):
    """
       Input:
            B: int cantidad de simulaciones a realizar
            lista_equipos: lista de m equipos participantes en la liga

       Output:
            Matriz mxm de probabilidades de cada equipo de quedar en el puesto i-ésimo en la liga

       Description:
            Devuelve una matriz mxm de probabilidades de cada equipo de quedar en el puesto i-ésimo en la liga
    """

    # Inicializar matriz de conteo
    m = len(lista_equipos)
    conteo = [[0 for _ in range(m)] for _ in range(m)]

    for _ in range(B):
        # Asumimos que simular_liga devuelve la lista de equipos en orden de llegada
        resultado_liga = simular_liga(lista_equipos, params)[1]
        for posicion, equipo in enumerate(resultado_liga):
            idx_equipo = lista_equipos.index(equipo)
            conteo[idx_equipo][posicion] += 1

    # Convertir a diccionario de probabilidades
    probabilidades = {}
    for i, equipo in enumerate(lista_equipos):
        probabilidades[equipo] = [valor / B for valor in conteo[i]]

    return probabilidades

- Función para mostrar los resultados de forma elegante.

In [ ]:
def mostrar_heatmap(probabilidades: dict, titulo: str = "Premier League — probabilidades de clasificación", simulaciones: int = 1000):
    """
    Muestra un heatmap interactivo de probabilidades de clasificación.

    Input:
        probabilidades: dict  → output directo de estimar_liga()
                                {"Arsenal": [0.6, 0.3, ...], "Man City": [...], ...}
        titulo:         str   → título del gráfico
        simulaciones:   int   → número de simulaciones usado (solo para el subtítulo)
    """

    data_json = json.dumps(probabilidades)

    html = f"""
<!DOCTYPE html>
<html>
<head>
<style>
  #hm-wrap {{ padding: 1rem 0; font-family: sans-serif; }}
  #hm-wrap h3 {{ font-size: 15px; font-weight: 500; color: #111; margin: 0 0 4px 0; }}
  #hm-subtitle {{ font-size: 12px; color: #888; margin: 0 0 16px 0; }}
  #hm-scroll {{ overflow-x: auto; }}
  table#hm {{ border-collapse: collapse; width: 100%; min-width: 700px; }}
  table#hm th {{ font-size: 11px; font-weight: 400; color: #888; text-align: center; padding: 2px 3px; width: 36px; }}
  table#hm td.team-name {{ font-size: 12px; color: #111; white-space: nowrap; padding: 2px 10px 2px 0; text-align: right; min-width: 110px; }}
  table#hm td.cell {{ width: 36px; height: 28px; text-align: center; vertical-align: middle; font-size: 10px; font-weight: 500; border-radius: 3px; cursor: default; }}
  table#hm td.cell:hover {{ outline: 1.5px solid #333; }}
  #hm-legend {{ display: flex; align-items: center; gap: 8px; margin-top: 14px; font-size: 11px; color: #888; }}
  #hm-legend-bar {{ display: flex; height: 10px; width: 160px; border-radius: 3px; overflow: hidden; }}
  #hm-zones {{ display: flex; gap: 16px; margin-top: 10px; flex-wrap: wrap; }}
  .zone-tag {{ font-size: 11px; padding: 2px 8px; border-radius: 3px; }}
</style>
</head>
<body>
<div id="hm-wrap">
  <h3>{titulo}</h3>
  <p id="hm-subtitle">Basado en {simulaciones:,} simulaciones · Modelo Dixon-Coles · Hover para ver valor exacto</p>
  <div id="hm-scroll">
    <table id="hm"></table>
  </div>
  <div id="hm-legend">
    <span>0%</span>
    <div id="hm-legend-bar"></div>
    <span>100%</span>
    <span style="margin-left:8px;color:#aaa;">probabilidad de quedar en esa posición</span>
  </div>
  <div id="hm-zones"></div>
</div>

<script>
const raw = {data_json};

function probToColor(p) {{
  const dark = window.matchMedia('(prefers-color-scheme: dark)').matches;
  if (p === 0) return dark ? '#1a1a1a' : '#f5f5f2';
  const stops = dark
    ? ['#0c2a4a','#0e3d6e','#1260a8','#2a87d8','#64b8f5','#a8d8fb']
    : ['#e6f1fb','#b5d4f4','#85b7eb','#378add','#185fa5','#0c447c'];
  const idx = Math.min(stops.length - 1, Math.floor(p * stops.length));
  return stops[idx];
}}

function textColor(p) {{
  const dark = window.matchMedia('(prefers-color-scheme: dark)').matches;
  if (p === 0) return dark ? '#444' : '#ccc';
  if (p >= 0.4) return dark ? '#a8d8fb' : '#042c53';
  if (p >= 0.2) return dark ? '#64b8f5' : '#0c447c';
  return dark ? '#2a87d8' : '#185fa5';
}}

const teams = Object.keys(raw);
const n = raw[teams[0]].length;
const expectedPos = t => raw[t].reduce((s, p, i) => s + p * (i + 1), 0);
const sorted = [...teams].sort((a, b) => expectedPos(a) - expectedPos(b));

const table = document.getElementById('hm');

// Cabecera
const headRow = document.createElement('tr');
let headHtml = '<th style="text-align:right;padding-right:10px;font-size:11px;color:#888;">equipo</th>';
for (let i = 1; i <= n; i++) {{
  const bg = i <= 4 ? '#e6f1fb' : i <= 6 ? '#eaf3de' : i > n - 3 ? '#fcebeb' : 'transparent';
  headHtml += `<th style="background:${{bg}};border-radius:3px;">${{i}}</th>`;
}}
headRow.innerHTML = headHtml;
table.appendChild(headRow);

// Filas por equipo
sorted.forEach(team => {{
  const tr = document.createElement('tr');
  let html = `<td class="team-name">${{team}}</td>`;
  raw[team].forEach((p, i) => {{
    const pos = i + 1;
    const zone = pos <= 4 ? 'Champions' : pos <= 6 ? 'Europa' : pos > n - 3 ? 'Descenso' : '';
    const label = p > 0 ? Math.round(p * 100) + '%' : '';
    const title = `${{team}} — posición ${{pos}}${{zone ? ' (' + zone + ')' : ''}}: ${{Math.round(p*100)}}%`;
    html += `<td class="cell" style="background:${{probToColor(p)}};color:${{textColor(p)}};" title="${{title}}">${{label}}</td>`;
  }});
  tr.innerHTML = html;
  table.appendChild(tr);
}});

// Leyenda degradado
const bar = document.getElementById('hm-legend-bar');
for (let i = 0; i < 40; i++) {{
  const d = document.createElement('div');
  d.style.flex = '1';
  d.style.background = probToColor(i / 40);
  bar.appendChild(d);
}}

// Zonas
const zoneData = [
  {{ label: 'Champions (1-4)',    bg: '#e6f1fb', fg: '#185fa5' }},
  {{ label: 'Europa (5-6)',       bg: '#eaf3de', fg: '#3b6d11' }},
  {{ label: `Descenso (${{n-2}}-${{n}})`, bg: '#fcebeb', fg: '#a32d2d' }},
];
const zonesDiv = document.getElementById('hm-zones');
zoneData.forEach(z => {{
  const s = document.createElement('span');
  s.className = 'zone-tag';
  s.style.background = z.bg;
  s.style.color = z.fg;
  s.textContent = z.label;
  zonesDiv.appendChild(s);
}});
</script>
</body>
</html>
"""
    display(HTML(html))

### **2.2. PREDICCIONES 24/25.**

Cargamos la lista de equipos de la Premier League de la temporada 24/25.

In [ ]:
equipos_2425 = [
    "Arsenal",
    "Aston Villa",
    "Bournemouth",
    "Brentford",
    "Brighton",
    "Chelsea",
    "Crystal Palace",
    "Everton",
    "Fulham",
    "Ipswich",
    "Leicester",
    "Liverpool",
    "Man City",
    "Man United",
    "Newcastle",
    "Nott'm Forest",
    "Southampton",
    "Tottenham",
    "West Ham",
    "Wolves"
]

In [ ]:
simular_partido("Arsenal", "Man City", params2324)

'1-2'

In [ ]:
estimar_partido("Arsenal", "Man City", params2324)

('1-1', 0.403, 0.252, 0.345)

In [ ]:
simular_liga(equipos_2425, params2324)[0]

{'Liverpool': 91,
 'Man City': 91,
 'Arsenal': 82,
 'Aston Villa': 67,
 'Tottenham': 64,
 'Newcastle': 63,
 'Chelsea': 58,
 'Man United': 56,
 'Bournemouth': 52,
 'Crystal Palace': 47,
 'Fulham': 46,
 "Nott'm Forest": 46,
 'Brentford': 45,
 'Brighton': 43,
 'West Ham': 42,
 'Wolves': 42,
 'Leicester': 36,
 'Everton': 35,
 'Southampton': 30,
 'Ipswich': 23}

In [ ]:
sim = 100
probabilidades = estimar_liga(B=sim, lista_equipos=equipos_2425, params=params2324)
mostrar_heatmap(probabilidades, simulaciones=sim)

### **2.3. PREDICCIONES 25/26.**

Cargamos la lista de equipos de la Premier League de la temporada 25/26.

In [ ]:
equipos_2526 = [
    "Arsenal",
    "Aston Villa",
    "Bournemouth",
    "Brentford",
    "Brighton",
    "Burnley",
    "Chelsea",
    "Crystal Palace",
    "Everton",
    "Fulham",
    "Leeds",
    "Liverpool",
    "Man City",
    "Man United",
    "Newcastle",
    "Nott'm Forest",
    "Sunderland",
    "Tottenham",
    "West Ham",
    "Wolves"
]

In [ ]:
simular_partido("Arsenal", "Man City", params2425)

'3-4'

In [ ]:
estimar_partido("Arsenal", "Man City", params2425)

('1-1', 0.404, 0.283, 0.313)

In [ ]:
simular_liga(equipos_2526, params2425)[0]

{'Liverpool': 89,
 'Arsenal': 81,
 'Newcastle': 80,
 'Man City': 79,
 'Chelsea': 71,
 "Nott'm Forest": 61,
 'Man United': 60,
 'Crystal Palace': 58,
 'Aston Villa': 57,
 'Tottenham': 53,
 'Brentford': 50,
 'Bournemouth': 47,
 'Fulham': 46,
 'Everton': 44,
 'West Ham': 41,
 'Brighton': 40,
 'Burnley': 37,
 'Wolves': 36,
 'Sunderland': 25,
 'Leeds': 19}

In [ ]:
sim = 100
probabilidades = estimar_liga(B=sim, lista_equipos=equipos_2526, params=params2425)
mostrar_heatmap(probabilidades, simulaciones=sim)

### **2.4. PREDICCIONES PARA UNA SOLA JORNADA. APUESTAS.**

En esta sección se evalúa la capacidad predictiva del modelo mediante un ejercicio de validación retrospectiva (backtesting). Para ello, se utilizan los parámetros estimados a partir de la temporada 2024/2025 para predecir los resultados de las jornadas 1 a 19 de la temporada 2025/2026. Dado que el modelo proporciona probabilidades para los tres posibles desenlaces de cada partido (victoria local, empate o victoria visitante), se procede a transformar esta información probabilística en una predicción puntual seleccionando, en cada caso, el resultado con mayor probabilidad estimada (criterio de máxima verosimilitud o máximo a posteriori en este contexto discreto).

A partir de estas predicciones, se construye una “apuesta conjunta” sobre todos los partidos de la jornada. La probabilidad teórica de dicha apuesta se calcula como el producto de las probabilidades individuales asociadas a cada resultado seleccionado, bajo el supuesto de independencia entre partidos.

Finalmente, se comparan las predicciones del modelo con los resultados reales observados, codificados de forma categórica (1 para victoria local, X para empate y 2 para victoria visitante). Esta comparación permite identificar aciertos y errores mediante una correspondencia exacta entre la predicción y el resultado observado. En conjunto, este procedimiento permite evaluar tanto la coherencia probabilística de las predicciones como su capacidad efectiva para anticipar resultados en un entorno fuera de muestra.

In [ ]:
# ============================================================
# BLOQUE 1: DATOS — Premier League 2025/26, Jornadas 1-19
# ============================================================
# Resultados reales de las jornadas 1 a 19, usados como conjunto de test
# para el backtesting del apartado 2.4. Se cargan desde un JSON aparte
# para no saturar el notebook con datos.

with open('jornadas_2025_26.json', encoding='utf-8') as f:
    jornadas = {int(k): v for k, v in json.load(f).items()}


In [ ]:
# ============================================================
# BLOQUE 2: CÁLCULOS
# ============================================================

random.seed(42)

def prob_a_apuesta(prob_local, prob_empate, prob_visitante):
    """Devuelve la apuesta ('1','X','2') con mayor probabilidad y su prob."""
    opciones = [("1", prob_local), ("X", prob_empate), ("2", prob_visitante)]
    return max(opciones, key=lambda x: x[1])

MC_ITERACIONES = 1000

resultados_jornadas = {}

for num_jornada, partidos in jornadas.items():
    prob_apuesta_jornada = 1.0
    aciertos_jornada     = 0
    detalle_partidos     = []

    for p in partidos:
        local          = p["local"]
        visitante      = p["visitante"]
        resultado_real = p["resultado"]

        # Llamada a estimar_partido con params2425
        _, prob_l, prob_e, prob_v = estimar_partido(
            local, visitante, params2425, MC=MC_ITERACIONES
        )

        apuesta, prob_elegida = prob_a_apuesta(prob_l, prob_e, prob_v)
        es_acierto = (apuesta == resultado_real)
        if es_acierto:
            aciertos_jornada += 1

        prob_apuesta_jornada *= prob_elegida

        detalle_partidos.append({
            "local":        local,
            "visitante":    visitante,
            "resultado":    resultado_real,
            "apuesta":      apuesta,
            "prob_1":       prob_l,
            "prob_X":       prob_e,
            "prob_2":       prob_v,
            "prob_elegida": prob_elegida,
            "acierto":      es_acierto,
        })

    resultados_jornadas[num_jornada] = {
        "partidos":       detalle_partidos,
        "prob_apuesta":   round(prob_apuesta_jornada, 12),
        "aciertos":       aciertos_jornada,
        "total_partidos": len(partidos),
    }

# ── Métricas globales ────────────────────────────────────────────────────────
probs_apuesta   = [v["prob_apuesta"] for v in resultados_jornadas.values()]
aciertos_list   = [v["aciertos"]     for v in resultados_jornadas.values()]

prob_media_global     = round(sum(probs_apuesta) / len(probs_apuesta), 12)
aciertos_media_global = round(sum(aciertos_list) / len(aciertos_list), 4)

jornada_max_prob = max(resultados_jornadas, key=lambda j: resultados_jornadas[j]["prob_apuesta"])
jornada_min_prob = min(resultados_jornadas, key=lambda j: resultados_jornadas[j]["prob_apuesta"])

print("✅ Bloque 2 completado")
print(f"   Probabilidad media de apuesta (19 jornadas): {prob_media_global:.4e}")
print(f"   Aciertos medios por jornada:                 {aciertos_media_global:.2f}")
print(f"   Jornada con MAYOR prob. apuesta: {jornada_max_prob}")
print(f"   Jornada con MENOR prob. apuesta: {jornada_min_prob}")

✅ Bloque 2 completado
   Probabilidad media de apuesta (19 jornadas): 1.4969e-03
   Aciertos medios por jornada:                 5.00
   Jornada con MAYOR prob. apuesta: 16
   Jornada con MENOR prob. apuesta: 5


In [ ]:
# ============================================================
# BLOQUE 3: VISUALIZACIÓN DE RESULTADOS
# ============================================================

def separador(char="─", n=76):
    print(char * n)

def pct(p):
    return f"{p*100:.2f}%"

separador("═")
print("   BACKTESTING — PREMIER LEAGUE 2025/26  ·  JORNADAS 1 A 19")
separador("═")

print(f"\n  📊 MÉTRICAS GLOBALES")
separador()
print(f"  {'Probabilidad media de apuesta conjunta':46s}: {prob_media_global:.4e}")
print(f"  {'Aciertos medios por jornada':46s}: {aciertos_media_global:.2f} / 10")
separador()

# ── Tabla resumen por jornada ────────────────────────────────────────────────
print(f"\n  {'JORNADA':^9} {'PARTIDOS':^9} {'ACIERTOS':^12} {'PROB. CONJUNTA':^20}")
separador()
for j in range(1, 20):
    d = resultados_jornadas[j]
    marca = ""
    if j == jornada_max_prob: marca = "  ◀ MÁX PROB"
    if j == jornada_min_prob: marca = "  ◀ MÍN PROB"
    print(f"  Jornada {j:2d}   {d['total_partidos']:^9d} "
          f"  {d['aciertos']:2d}/{d['total_partidos']:<7d}"
          f"  {d['prob_apuesta']:.4e}{marca}")
separador()

# ── Detalle: jornada con MAYOR probabilidad ──────────────────────────────────
print(f"\n  🏆 JORNADA CON MAYOR PROBABILIDAD → Jornada {jornada_max_prob}")
separador()
dm = resultados_jornadas[jornada_max_prob]
print(f"  Prob. apuesta conjunta : {dm['prob_apuesta']:.4e}")
print(f"  Aciertos               : {dm['aciertos']} / {dm['total_partidos']}\n")
print(f"  {'LOCAL':<16} {'VISITANTE':<16} {'REAL':^5} {'APUESTA':^7} "
      f"{'P(1)':^7} {'P(X)':^7} {'P(2)':^7} {'P.ELEG':^8} {'OK':^4}")
separador("-")
for p in dm["partidos"]:
    ok = "✓" if p["acierto"] else "✗"
    print(f"  {p['local']:<16} {p['visitante']:<16} {p['resultado']:^5} "
          f"{p['apuesta']:^7}  {pct(p['prob_1']):^7} {pct(p['prob_X']):^7} "
          f"{pct(p['prob_2']):^7} {pct(p['prob_elegida']):^9} {ok:^4}")
separador()

# ── Detalle: jornada con MENOR probabilidad ──────────────────────────────────
print(f"\n  📉 JORNADA CON MENOR PROBABILIDAD → Jornada {jornada_min_prob}")
separador()
di = resultados_jornadas[jornada_min_prob]
print(f"  Prob. apuesta conjunta : {di['prob_apuesta']:.4e}")
print(f"  Aciertos               : {di['aciertos']} / {di['total_partidos']}\n")
print(f"  {'LOCAL':<16} {'VISITANTE':<16} {'REAL':^5} {'APUESTA':^7} "
      f"{'P(1)':^7} {'P(X)':^7} {'P(2)':^7} {'P.ELEG':^8} {'OK':^4}")
separador("-")
for p in di["partidos"]:
    ok = "✓" if p["acierto"] else "✗"
    print(f"  {p['local']:<16} {p['visitante']:<16} {p['resultado']:^5} "
          f"{p['apuesta']:^7}  {pct(p['prob_1']):^7} {pct(p['prob_X']):^7} "
          f"{pct(p['prob_2']):^7} {pct(p['prob_elegida']):^9} {ok:^4}")
separador("═")

════════════════════════════════════════════════════════════════════════════
   BACKTESTING — PREMIER LEAGUE 2025/26  ·  JORNADAS 1 A 19
════════════════════════════════════════════════════════════════════════════

  📊 MÉTRICAS GLOBALES
────────────────────────────────────────────────────────────────────────────
  Probabilidad media de apuesta conjunta        : 1.4969e-03
  Aciertos medios por jornada                   : 5.00 / 10
────────────────────────────────────────────────────────────────────────────

   JORNADA  PARTIDOS    ACIERTOS      PROB. CONJUNTA   
────────────────────────────────────────────────────────────────────────────
  Jornada  1      10        5/10       1.3656e-03
  Jornada  2      10        5/10       9.2117e-04
  Jornada  3      10        5/10       9.5916e-04
  Jornada  4      10        7/10       2.0728e-03
  Jornada  5      10        2/10       6.3581e-04  ◀ MÍN PROB
  Jornada  6      10        4/10       1.9124e-03
  Jornada  7      10        7/10       1.3

## **3. SIMULACIÓN DE LA PREMIER LEAGUE SPARK**
En esta sección vamos a tratar de paralelizar algunas de las funciones previas: estimar_partido, simular_liga y estimar_liga. Spark permite ejecutar código en paralelo lo que es útil para trabajar con grandes cantidades de datos que no se pueden manejar en un solo ordenador por memoria (como hemos visto en clase) o para optimizar procesos como pueden ser en este caso las simulaciones. En el apartado 4 compararemos tiempos y justificaremos la aplicación de Spark para esta tarea.

Los cambios óptimos que proponemos son:
- Paralelizar simular_partido en estimar_partido, en total MC paralelizaciones.
- Paralelizar cada partido de una liga en simular_liga, en total Nx(N-1), en este caso N = 20.
- Paralelizar cada simulación de liga en estimar_liga.

In [ ]:
spark = (
    SparkSession.builder
    .appName('PremierLeague_DixonColes')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config("spark.default.parallelism", 8)
    .config("spark.sql.shuffle.partitions", 8)
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel('WARN')
print(f'Spark {spark.version} · cores disponibles: {sc.defaultParallelism}')

Spark 4.0.2 · cores disponibles: 8


In [ ]:
def estimar_partido_spark(equipo_local, equipo_visitante, params, MC=1000):

    params_bc = sc.broadcast(params)

    resultados = sc.parallelize(range(MC), numSlices = 8) \
        .map(lambda _: simular_partido(
            equipo_local,
            equipo_visitante,
            params_bc.value
        )) \
        .collect()

    params_bc.unpersist()

    glocal = sum(int(r.split('-')[0]) > int(r.split('-')[1]) for r in resultados)
    gvisit = sum(int(r.split('-')[0]) < int(r.split('-')[1]) for r in resultados)

    prob_local = round(glocal / MC, 4)
    prob_visitante = round(gvisit / MC, 4)
    prob_empate = round(1 - prob_local - prob_visitante, 4)

    moda = str(random.choice(statistics.multimode(resultados)))

    return moda, prob_local, prob_empate, prob_visitante

In [ ]:
def simular_liga_spark(equipos, params):

    params_bc = sc.broadcast(params)

    partidos = [(h, a) for h in equipos for a in equipos if h != a]

    resultados = sc.parallelize(partidos, numSlices = 64) \
        .map(lambda par: (
            par[0],
            par[1],
            simular_partido(par[0], par[1], params_bc.value)
        )) \
        .collect()

    params_bc.unpersist()

    puntos = {e: 0 for e in equipos}

    for h, a, marcador in resultados:
        gl, gv = map(int, marcador.split('-'))

        if gl > gv:
            puntos[h] += 3
        elif gl < gv:
            puntos[a] += 3
        else:
            puntos[h] += 1
            puntos[a] += 1

    ordenados = dict(sorted(puntos.items(), key=lambda x: x[1], reverse=True))

    return ordenados, list(ordenados.keys())

In [ ]:
def simular_clasificacion_spark(equipos, params):

    puntos = {e: 0 for e in equipos}

    for h in equipos:
        for a in equipos:
            if h == a:
                continue

            resultado = simular_partido(h, a, params)
            gl, gv = map(int, resultado.split('-'))

            if gl > gv:
                puntos[h] += 3
            elif gl < gv:
                puntos[a] += 3
            else:
                puntos[h] += 1
                puntos[a] += 1

    return sorted(puntos, key=puntos.get, reverse=True)

def estimar_liga_spark(B=1000, lista_equipos=None, params=None):

    m = len(lista_equipos)

    params_bc = sc.broadcast(params)
    equipos_bc = sc.broadcast(lista_equipos)

    # Simulación de B ligas en paralelo
    resultados = sc.parallelize(range(B), numSlices = 64) \
        .map(lambda _: simular_clasificacion_spark(
            equipos_bc.value,
            params_bc.value
        )) \
        .collect()

    params_bc.unpersist()
    equipos_bc.unpersist()

    # Conteo de posiciones por equipo
    conteo = {e: [0] * m for e in lista_equipos}

    for clasificacion in resultados:
        for pos, equipo in enumerate(clasificacion):
            conteo[equipo][pos] += 1

    # Normalización a probabilidades
    probabilidades = {
        e: [v / B for v in conteo[e]]
        for e in lista_equipos
    }

    return probabilidades

## **4. COMPARACIÓN DE TIEMPOS**

In [ ]:
def medir_tiempo(func, *args, **kwargs):
    t0 = time.perf_counter()
    res = func(*args, **kwargs)
    t1 = time.perf_counter()
    return res, t1 - t0

In [ ]:
MC = 5000

_, t_seq = medir_tiempo(
    estimar_partido,
    "Arsenal", "Man City", params2425, MC
)

_, t_spark = medir_tiempo(
    estimar_partido_spark,
    "Arsenal", "Man City", params2425, MC
)

print("Partido Secuencial:", t_seq)
print("Partido Spark:", t_spark)

Partido Secuencial: 125.56001233300049
Partido Spark: 121.22637854299865


In [ ]:
B = 200 # Son pocas simulaciones y ya tarda 1 hora

# Secuencial
_, t_seq = medir_tiempo(
    lambda: estimar_liga(B=B, lista_equipos=equipos_2526, params=params2425)
)

# Spark
_, t_spark = medir_tiempo(
    estimar_liga_spark,
    B, equipos_2526, params2425
)

print("Secuencial:", t_seq)
print("Spark:", t_spark)

Secuencial: 1920.632375258001
Spark: 1719.3877370090013


## **5. CONCLUSIÓN**

Este trabajo ha combinado el modelo estadístico de Dixon-Coles con PySpark para abordar la predicción de resultados en la Premier League desde dos ángulos: la calidad del modelo y la eficiencia computacional.

En cuanto al modelo, la estimación de parámetros es sensible al volumen de datos históricos. Incorporar una temporada adicional al entrenamiento duplica el tiempo de estimación, lo que ilustra un compromiso clásico entre riqueza de información y coste computacional. La ponderación temporal exponencial del modelo mitiga parcialmente este problema al reducir el peso de los partidos más antiguos, pero no elimina la necesidad de procesarlos.

En cuanto a la escalabilidad, la versión Spark demuestra que el enfoque es directamente extrapolable a escenarios de mayor complejidad: añadir nuevas competiciones, ligas de otros países o incluso extender el modelo a estadísticas de jugadores individuales; donde el número de entidades a estimar crece considerablemente, no requiere rediseñar la arquitectura. Esta propiedad de escalabilidad horizontal es precisamente donde Spark aporta mayor valor frente a una implementación secuencial.

En conjunto, Dixon-Coles con Spark ofrece un marco robusto, interpretable y eficiente para la predicción deportiva, con margen real de extensión tanto en profundidad estadística como en cobertura de datos.

## **6. BIBLIOGRAFÍA**

Paper original de Dixon-Coles: https://www.ajbuckeconbikesail.net/wkpapers/Airports/MVPoisson/soccer_betting.pdf

Artículo seguido (Descripción del modelo + Código para la obtención de los parámetros):  https://dashee87.github.io/football/python/predicting-football-results-with-statistical-modelling-dixon-coles-and-time-weighting/

Bases de datos: https://www.football-data.co.uk/englandm.php